### FAISS 
https://github.com/facebookresearch/faiss

FAISS is a library for efficient similarity search and clustering of dense vectors.

Key advantages:
1. Extremely fast similarity search
2. Memory efficient
3. Supports GPU acceleration
4. Can handle millions of vectors

How it works:
- Indexes vectors for fast nearest neighbor search
- Returns most similar vectors based on distance metrics


In [45]:
#load libraries

import os
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings('ignore')

#langchain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage,AIMessage

#langchain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain



In [89]:
import os
load_dotenv()
os.getenv("GROQ_API_KEY")

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

### Data Ingestion and Processing

In [47]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

In [48]:
#text splitting
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=150,
    length_function=len,
    separators=[" "]
)

#split the doc into chunks
chunks=text_splitter.split_documents(sample_documents)
print(chunks[0])
print(chunks[1])

page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
page_content='Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.' metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}


In [49]:
print(f"Created {len(chunks)} chunks")
print(f"Content : {chunks[0].page_content}")
print(f"Metadata : {chunks[0].metadata}")

Created 4 chunks
Content : Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
Metadata : {'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [50]:
# Initialize HuggingFace embeddings with the latest model


sample_text="MAchine LEARNing is fascinating"
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)
embeddings

## Example: create a embedding for a single text
sample_text="What is machine learning"
sample_embedding=embeddings.embed_query(sample_text)
sample_embedding

[-0.01427509170025587,
 0.0230745617300272,
 -0.057932350784540176,
 -0.011700417846441269,
 0.004516063258051872,
 0.034621115773916245,
 -0.009407958947122097,
 0.010936843231320381,
 -0.012402592226862907,
 0.003968873061239719,
 0.07477064430713654,
 0.06507954001426697,
 -0.03539096564054489,
 0.04704267904162407,
 0.054117847234010696,
 -0.05851410701870918,
 0.028937699273228645,
 -0.041821159422397614,
 0.01638462021946907,
 -0.015412762761116028,
 -0.0457816980779171,
 -0.04141760990023613,
 0.004164328332990408,
 0.008351932279765606,
 -0.04056616872549057,
 -0.042050451040267944,
 -0.0027098525315523148,
 -0.027915023267269135,
 -0.040728919208049774,
 0.02508067712187767,
 -0.005217349156737328,
 -0.040359411388635635,
 -0.008940053172409534,
 0.052906472235918045,
 1.6607117458988796e-06,
 -0.033687472343444824,
 -0.0189888384193182,
 0.011894340626895428,
 -0.027248546481132507,
 -0.03066549263894558,
 0.04450800642371178,
 0.0066995397210121155,
 0.0222290251404047,
 0.0

In [51]:
texts=["AI","MAchine learning","Deep Learning","Neural Network"]
batch_embeddings=embeddings.embed_documents(texts)
print(batch_embeddings[0])

[0.008028431795537472, 0.04928513616323471, -0.053655147552490234, -0.009137765504419804, -0.02992810681462288, 0.02027035690844059, 0.0076807597652077675, 0.015737373381853104, 0.014865539036691189, 0.005490824114531279, 0.05760718509554863, -0.009031164459884167, -0.058842405676841736, 0.06809139996767044, 0.03182908892631531, -0.03633217513561249, 0.020421059802174568, -0.020537281408905983, 0.027460722252726555, -0.023199332877993584, -0.030417446047067642, -0.012269556522369385, -0.015492553822696209, 0.014838089235126972, -0.025529541075229645, -0.026232635602355003, -0.030431021004915237, -0.03571871668100357, 0.00816231220960617, 0.00902101956307888, -0.024465598165988922, -0.037079308182001114, -0.033491503447294235, 0.01860124245285988, 1.875937527984206e-06, -0.02887306921184063, 0.004642372950911522, -0.0051481351256370544, -0.017725450918078423, -0.034275561571121216, -0.0058472538366913795, 0.056785114109516144, -0.007765411399304867, 0.01901083067059517, -0.0584707185626

In [52]:
#Compare Embeddings with similarity search
def compare_emb(text1:str,text2:str):
    """Compare embeddings of two strings"""
    emb1=np.array(embeddings.embed_query(text1))
    emb2=np.array(embeddings.embed_query(text2))

    similarity=np.dot(emb1,emb2) / (np.linalg.norm(emb1)*np.linalg.norm(emb2))
    return similarity

In [53]:
print(f"AI VS Aritifical Intelligence : {compare_emb("AI","Artificial Intelligence"):.3f}")

AI VS Aritifical Intelligence : 0.788


In [54]:
print(f"AI VS Pizza:{compare_emb("AI","Pizza"):.3f}")

AI VS Pizza:0.272


### Create FAISS Vector Store

In [55]:
vectorstore=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"Vector Store created w {vectorstore.index.ntotal} vectors")

Vector Store created w 4 vectors


In [56]:
vectorstore

In [57]:
#save vector store for later use

vectorstore.save_local("faiss_index") #saving in hard disk
print("Vector store saved to 'faisee index' directory")

Vector store saved to 'faisee index' directory


In [58]:
#load vector store
loaded_vectorstore=FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)


print(f"Loaded vectore store contains {loaded_vectorstore.index.ntotal} vectors")

Loaded vectore store contains 4 vectors


In [59]:
#Similarity search
query="What is deep learning"

results=vectorstore.similarity_search(query,k=3)
print(results)

[Document(id='7af7c224-7d11-4998-9d21-034d6a749fce', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='b38a6a40-3ad0-4ad2-84e1-1013e8ea6241', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='15c338fc-9e61-4b41-84c3-a3dd7ccb1bcb', metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        T

In [60]:
print(f"Query:{query}\n")
print("Top 3 similar chunks:")
for i,doc in enumerate(results):
    print(f"\n{i+1}. Source : {doc.metadata['source']}")
    print(f"         Content:{doc.page_content[:500]}...")

Query:What is deep learning

Top 3 similar chunks:

1. Source : Deep Learning
         Content:Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition....

2. Source : ML Basics
         Content:Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning....

3. Source : AI Introduction
         Content:Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI....


In [61]:
#Similarity Search with score

results_w_scores=vectorstore.similarity_search_with_score(query,k=3)

print("\n\nSimilariy Search w scores:")
for doc,score in results_w_scores:
    print(f"\nSource:{score:.3f}")
    print(f"\nSource:{doc.metadata['source']}")
    print(f"Content preview:{doc.page_content[:100]}...")



Similariy Search w scores:

Source:0.267

Source:Deep Learning
Content preview:Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses m...

Source:0.863

Source:ML Basics
Content preview:Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being...

Source:1.147

Source:AI Introduction
Content preview:Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These syst...


- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)

In [62]:
chunks

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.'),
 Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'),
 Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recogn

In [63]:
# Search w metadata filtering
filter_dict={"topic":"ML"}
filtered_results=vectorstore.similarity_search(
    query,
    k=3,
    filter=filter_dict
)
print(filtered_results)

[Document(id='b38a6a40-3ad0-4ad2-84e1-1013e8ea6241', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.')]


In [64]:
len(filtered_results)

1

### Build RAG Chain with LCEL

In [ ]:
#LLM GROQ LLM

from langchain.chat_models import init_chat_model
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")

llm=init_chat_model(model="groq:llama-3.1-8b-instant")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001E3C2D69E50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E3C2D6A850>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [66]:
#Simple RAG Chain with LCEL

simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [67]:
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [68]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E3C2D69590>, search_kwargs={'k': 3})

In [69]:
#Format documents for the prompt
from typing import List
def format_docs(docs:List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted=[]
    for i,doc in enumerate(docs):
        source=doc.metadata.get('source','Unknown')
        formatted.append(f"Document {i+1} (Source:{source}):\n{doc.page_content}")
        return "\n\n".join(formatted)
    

#Each Document has metadata (like file name, URL, or source).
#If 'source' is missing, it defaults to 'Unknown'.

In [70]:
simple_rag_chain=(
    {"context":retriever | format_docs, "question":RunnablePassthrough()}
    | simple_prompt
    | llm
    | StrOutputParser()
)

{"context": retriever | format_docs, "question": RunnablePassthrough()}

Defines inputs to the chain.

retriever gets relevant documents based on the user query.

format_docs formats those docs into readable text.

RunnablePassthrough() passes the user’s question directly as "question".

| simple_prompt

Combines the formatted context and question into a prompt template for the LLM.

| llm

Sends that prompt to the language model (e.g., GPT) to generate an answer.

| StrOutputParser()

Converts the LLM’s structured output into plain text.

In [71]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E3C2D69590>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001E3C2D69E50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E3C2D6A850>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))
| StrOutputParser()

In [72]:
# Conversational rag chain
conversational_prompt=ChatPromptTemplate.from_messages([
    ("system","you are a helpful AI Assistant. Use the provided context to answer the questions."),
    ("placeholder","{chat_history}"),
    ("human","Context:{context}\n\nQuestion:{input}"),
])

In [84]:
def create_conversational_rag():
    """Create a conversational RAG chain with emory"""
    return (
        RunnablePassthrough.assign(
            context=lambda x:format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )
conversational_rag=create_conversational_rag()

In [74]:
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(s['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag=

In [75]:
#Streaming RAG Chain
streaming_rag_chain=(
    {"context":retriever | format_docs, "question":RunnablePassthrough()}
    | simple_prompt
    | llm
)
streaming_rag=streaming_rag_chain

In [76]:
streaming_rag

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001E3C2D69590>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001E3C2D69E50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E3C2D6A850>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

Modern RAG chains created successfully!
Available chains:
- simple_rag_chain: Basic Q&A
- conversational_rag: Maintains conversation history
- streaming_rag_chain: Supports token streaming

In [77]:
# Test function for different chain types
def test_rag_chains(question: str):
    """Test all RAG chain variants"""
    print(f"Question: {question}")
    print("=" * 80)
    
    # 1. Simple RAG
    print("\n1. Simple RAG Chain:")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")

    # 2. Streaming RAG
    print("\n2. Streaming RAG:")
    print("Answer: ", end="", flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end="", flush=True)
    print()

In [78]:
test_rag_chains("What is the difference between AI & machine learning?")

Question: What is the difference between AI & machine learning?

1. Simple RAG Chain:
Answer: Machine Learning is a subset of AI. This means AI encompasses Machine Learning but also includes other techniques or approaches that enable systems to perform tasks without being explicitly programmed.

In simpler terms, AI is a broader field that includes Machine Learning as one of its subsets.

2. Streaming RAG:
Answer: According to the context, the main difference between AI and machine learning is that AI is a broader field, and machine learning is a subset of AI. In other words, machine learning is a specific technique used in AI that enables systems to learn from data.


In [79]:
# Test with multiple questions
test_questions = [
    "What is the difference between AI and Machine Learning?",
    "Explain deep learning in simple terms",
    "How does NLP work?"
]

for question in test_questions:
    print("\n" + "=" * 80 + "\n")
    test_rag_chains(question)



Question: What is the difference between AI and Machine Learning?

1. Simple RAG Chain:
Answer: According to the context, the difference between AI and Machine Learning is that Machine Learning is a subset of AI.

2. Streaming RAG:
Answer: According to the context, the difference between AI and Machine Learning is that Machine Learning is a subset of AI.


Question: Explain deep learning in simple terms

1. Simple RAG Chain:
Answer: Deep learning is a type of artificial intelligence that helps computers understand and interpret data. It works by using multiple layers to break down raw information into its most important parts. Think of it like a hierarchy:

- First layer looks at the basics (like colors and shapes)
- Second layer looks at patterns and relationships
- Third layer looks at more complex concepts and meanings

This process allows computers to learn and understand information in a way that's similar to how humans do, making it useful for tasks like image recognition, lang

In [85]:
#Conversational RAG

print("\n3. Conversational RAG")
chat_history=[]

q1="What is machine learning?"
a1=conversational_rag.invoke({
    "input":q1,
    "chat_history":chat_history
})

print(f"Q1:{q1}")
print(f"A1:{a1}")


3. Conversational RAG
Q1:What is machine learning?
A1:According to Document 1, Machine Learning is a subset of AI that enables systems to learn from data.


In [87]:
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])

In [88]:
#Follow up question

q2="how is it different from traditioanl programming?"
a2=conversational_rag.invoke({
    "input":q2,
    "chat_history":chat_history
})
print(q2)
print(a2)

how is it different from traditioanl programming?
Machine learning is different from traditional programming in that it involves training algorithms on data to enable them to make predictions, classify objects, or make decisions without being explicitly programmed for each specific task.

In traditional programming, a developer writes specific code to solve a problem. The program is designed to perform a set of predefined tasks, and it relies on the developer's expertise and knowledge to make decisions.

Machine learning, on the other hand, uses algorithms to analyze data, learn patterns, and make predictions or decisions based on that data. The algorithm is trained on a dataset, and it can adapt to new data over time, allowing it to improve its performance and accuracy.

This approach allows machine learning models to:

1. Learn from data: Machine learning models can learn from large datasets, identifying patterns and relationships that may not be apparent to a human.
2. Make predicti